# Module 3: Generators and Decorators (Lab Streams)

**Concepts:** lazy evaluation · `yield` · generator expressions · decorators · retry

> **Safety note:** This is a **programming** exercise. It is **not** medical advice, not a diagnosis, and **not** for real patients or real medicines. We use invented drug names (for example **Alercin**) and school biology/chemistry only.

### How to use this notebook

1. Read the markdown cell (the story).
2. Run the code cell under it with **Shift+Enter**.
3. Check **What you should see**.
4. Try the exercises before opening **Solutions**.

**Remember:** Generators = **one piece at a time**. Decorators = **extra rules wrapped around** a function.

**Words**

| Word | Meaning |
|------|---------|
| **codon** | three DNA bases read together |
| **lazy** | compute the next item only when asked |
| **decorator** | a wrapper that adds logging or retry |

## 1. Lazy vs eager

**Eager:** build the full list now. **Lazy:** produce on demand (good for long DNA).

**What you should see:** a list `[1, 2, 3]` from both styles.

In [ ]:
eager = [1, 2, 3]
print("eager", eager)


def count_to_three():
    yield 1
    yield 2
    yield 3


print("lazy", list(count_to_three()))

## 2. `yield` — DNA codons one by one

Walk a sequence in steps of 3. Do not build every codon list if you only need the first few.

**What you should see:** codons `ATG`, `GCA`, `TTC`.

In [ ]:
def codons(dna):
    dna = dna.upper().replace(" ", "")
    i = 0
    while i + 3 <= len(dna):
        yield dna[i : i + 3]
        i += 3


for c in codons("ATGGCATTC"):
    print(c)

## 3. Generator expression — high pulse only

`(x for x in values if x > limit)` — parentheses, not square brackets.

**What you should see:** pulses above 100.

In [ ]:
pulses = [72, 88, 110, 95, 120, 80]
high = (p for p in pulses if p > 100)
print(list(high))

## 4. Small pipeline — parse vital lines lazily

Lines like `pulse=110` → keep high values.

**What you should see:** `[110, 120]`.

In [ ]:
lines = ["pulse=72", "pulse=110", "temp=37.2", "pulse=120"]


def parse_pulse(lines):
    for line in lines:
        if line.startswith("pulse="):
            yield int(line.split("=", 1)[1])


high = [v for v in parse_pulse(lines) if v > 100]
print(high)

## 5. Decorator — log CALL / DONE around dose

A decorator wraps a function without rewriting it.

**What you should see:** CALL, result, DONE.

In [ ]:
def log_calls(fn):
    def wrapper(*args, **kwargs):
        print("CALL", fn.__name__, args)
        result = fn(*args, **kwargs)
        print("DONE", fn.__name__, "->", result)
        return result

    return wrapper


@log_calls
def dose_mg(weight_kg, mg_per_kg=10, max_mg=1000):
    needed = weight_kg * mg_per_kg
    if needed > max_mg:
        return "OVER_MAX"
    return needed


dose_mg(52.5)

## 6. Retry — lab device busy

If the function raises, wait and try again (practice).

**What you should see:** fail once, then success.

In [ ]:
import time


def retry(times=3):
    def deco(fn):
        def wrapper(*args, **kwargs):
            last = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last = e
                    print(f"attempt {attempt} failed: {e}")
                    time.sleep(0.05)
            raise last

        return wrapper

    return deco


attempts = {"n": 0}


@retry(times=3)
def read_spectrometer():
    attempts["n"] += 1
    if attempts["n"] < 2:
        raise RuntimeError("device busy (practice)")
    return 0.42  # fake absorbance


print("result", read_spectrometer())

## 7. `functools.wraps` — keep the real name

**What you should see:** `__name__` is `dose_mg`, not `wrapper`.

In [ ]:
from functools import wraps


def log_calls(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print("CALL", fn.__name__)
        result = fn(*args, **kwargs)
        print("DONE")
        return result

    return wrapper


@log_calls
def dose_mg(weight_kg):
    return weight_kg * 10


print(dose_mg.__name__)
dose_mg(50)

## 8. Capstone — lazy vitals + logged max

**What you should see:** CALL/DONE around max pulse `120`.

In [ ]:
from functools import wraps


def log_calls(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print("CALL", fn.__name__)
        result = fn(*args, **kwargs)
        print("DONE", result)
        return result

    return wrapper


def pulse_values(lines):
    for line in lines:
        if line.startswith("pulse="):
            yield int(line.split("=", 1)[1])


@log_calls
def max_pulse(values):
    return max(values)


lines = ["pulse=72", "pulse=110", "pulse=120", "temp=37"]
print(max_pulse(list(pulse_values(lines))))

---
# Exercises

**Ex 1** — `countdown(n)` yields `n` … `1`.

In [ ]:
# TODO
# def countdown(n):
#     ...
# print(list(countdown(5)))

**Ex 2** — Generator expression: squares 1..5 → `[1, 4, 9, 16, 25]`.

In [ ]:
# TODO
# g = (...)
# print(list(g))

**Ex 3** — Decorator `upper_result`: if return value is `str`, return `.upper()`.

In [ ]:
# TODO
# @upper_result
# def ward():
#     return "a"
# print(ward())

**Ex 4** — `tail(lines, n)` yields the last `n` lines (list buffer OK).

In [ ]:
# TODO

---
# Solutions

In [ ]:
def countdown(n):
    while n >= 1:
        yield n
        n -= 1


print(list(countdown(5)))

In [ ]:
g = (i * i for i in range(1, 6))
print(list(g))

In [ ]:
def upper_result(fn):
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        if isinstance(result, str):
            return result.upper()
        return result

    return wrapper


@upper_result
def ward():
    return "a"


print(ward())

In [ ]:
def tail(lines, n):
    buf = list(lines)[-n:]
    for line in buf:
        yield line


print(list(tail(["a", "b", "c", "d"], 2)))

**Next:** Module 4 — errors and `with` (lab notebook always closes).